# Train on a free Colab GPU

Accepts **either** kind of zip:

- A dataset already exported by `backend/scripts/export_for_colab.py`
  (has `data.yaml` inside), **or**
- A plain zip of your own photos with hand-drawn red outlines on them —
  the same format the app's "Import pre-annotated images" feature accepts.
  This notebook extracts the outlines itself, right here on the GPU
  machine, using the exact same code the app uses. No 30MB chat-upload
  limit either — Colab's upload cell handles much larger files fine.

You can also optionally continue from a previous `best.pt` instead of
starting from stock COCO weights each time (see the "Optional: continue
from checkpoint" cell below) — same idea as the desktop app's "Continue
from checkpoint" option on the Train tab.

**Before running:** `Runtime` menu → `Change runtime type` → select **T4 GPU** → Save.

Then set `CLASS_NAME` below if needed, and `Runtime` → `Run all`. When it
gets to the upload cells, upload your zip (either kind), and optionally a
previous `best.pt` to continue from.

At the end, `trained-model.zip` (containing `best.pt`) downloads automatically —
that's what you bring back to **Detect → Import a model** (desktop app or web app).

In [ ]:
CLASS_NAME = "pin"  # only used if you upload a raw zip of red-annotated photos

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!pip install -q ultralytics opencv-python-headless pydantic

Clone the app's repo (just to reuse its red-outline-extraction code — nothing
else from it runs here).

In [ ]:
!rm -rf repo
!git clone -q --depth 1 -b claude/vision-model-object-detection-jbwh8s https://github.com/Robokks/object-detection-.git repo
import sys

sys.path.insert(0, "repo/backend")

In [ ]:
from google.colab import files

print("Upload a dataset zip (from export_for_colab.py) OR a plain zip of your red-annotated photos:")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.endswith(".zip")]
assert zip_names, "Expected a .zip file"
dataset_zip = zip_names[0]
print("Using:", dataset_zip)

Optionally continue fine-tuning from a **previous `best.pt`** (from an earlier Colab run, or exported from the desktop/web app) instead of starting over from stock COCO weights every time. When the upload dialog below appears, either pick a `.pt` file, or click **Cancel** to skip and start from stock weights.

In [ ]:
from google.colab import files

print("Optional: upload a previous best.pt to continue from (Cancel to skip):")
try:
    ckpt_upload = files.upload()
except Exception:
    ckpt_upload = {}
checkpoint_names = [n for n in ckpt_upload if n.endswith(".pt")]
BASE_CHECKPOINT = checkpoint_names[0] if checkpoint_names else None
print("Continuing from:", BASE_CHECKPOINT) if BASE_CHECKPOINT else print("Starting from stock yolov8n-seg.pt")

This cell figures out which kind of zip you uploaded and builds `dataset/`
either way. For a raw photo zip, it runs the same red-outline extraction +
red-line removal the app does, then an 80/20 train/val split.

In [ ]:
import random
import shutil
from pathlib import Path

shutil.unpack_archive(dataset_zip, "raw")
raw_root = Path("raw")
existing_yaml = next(raw_root.rglob("data.yaml"), None)

dataset_dir = Path("dataset")
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

if existing_yaml is not None:
    print("Found data.yaml — this is a pre-built dataset export, using it as-is.")
    shutil.copytree(existing_yaml.parent, dataset_dir)
else:
    print(f'No data.yaml found — treating this as raw red-annotated photos, class "{CLASS_NAME}".')
    from app.services import red_import_service

    image_files = sorted(
        p for p in raw_root.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    )
    assert image_files, "No image files found in the uploaded zip"
    print(f"Found {len(image_files)} image(s), extracting outlines…")

    random.seed(42)
    shuffled = image_files[:]
    random.shuffle(shuffled)
    val_count = max(1, int(len(shuffled) * 0.2)) if len(shuffled) > 1 else 0
    val_set = set(shuffled[:val_count])

    for split in ("train", "val"):
        (dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    total_shapes = 0
    for path in image_files:
        split = "val" if path in val_set else "train"
        content = path.read_bytes()
        try:
            clean_bytes, shapes = red_import_service.extract_annotated_image(content, CLASS_NAME)
        except Exception as e:
            print(f"  {path.name}: skipped ({e})")
            continue
        if not shapes:
            print(f"  {path.name}: no outlines found, skipped")
            continue

        stem = path.stem
        out_img = dataset_dir / "images" / split / f"{stem}.png"
        out_img.write_bytes(clean_bytes)

        from PIL import Image
        import io

        with Image.open(io.BytesIO(clean_bytes)) as im:
            width, height = im.size

        lines = []
        for shape in shapes:
            coords = []
            for px, py in shape.points:
                coords.append(f"{min(max(px / width, 0.0), 1.0):.6f}")
                coords.append(f"{min(max(py / height, 0.0), 1.0):.6f}")
            lines.append("0 " + " ".join(coords))
        (dataset_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        total_shapes += len(shapes)
        print(f"  {path.name}: {len(shapes)} shape(s) -> {split}")

    (dataset_dir / "data.yaml").write_text(
        f"path: .\ntrain: images/train\nval: images/val\nnames:\n  0: {CLASS_NAME}\n"
    )
    print(f"Total shapes extracted: {total_shapes}")

!echo '--- dataset/data.yaml ---'; cat dataset/data.yaml
!echo '--- image counts ---'; find dataset/images -type f | wc -l

## Train

Fine-tunes `yolov8n-seg` — the same architecture the app trains locally — starting from `BASE_CHECKPOINT` if you uploaded one above, otherwise from stock pretrained COCO weights (labels are polygon outlines, so the app always trains the segmentation variant, even for plain box labels).
Adjust `epochs`/`imgsz`/`batch` as needed; watch the `val` mask mAP in the output and stop early (interrupt the cell) if it plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_CHECKPOINT if BASE_CHECKPOINT else "yolov8n-seg.pt")
results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    project="runs",
    name="train",
    exist_ok=True,
)

In [ ]:
# quick sanity check on the validation set
metrics = model.val()
print(metrics.seg.map, "(mask mAP50-95)")

## Try the trained model + get full detection details as JSON

Optional — upload one or more test images to run the just-trained model on. Skip the upload (click **Cancel**) to use a few images from the validation split instead. Writes `colab-detections.json` with the same detail as the desktop app's Detect tab and `scripts/detect_to_json.py` (class, confidence, center position, orientation angle, left/right/top/bottom edge midpoints, and the bounding box for every detection) and downloads it.

In [ ]:
from pathlib import Path
from google.colab import files

print("Optional: upload image(s) to test the trained model on (Cancel to use validation images instead):")
try:
    test_upload = files.upload()
except Exception:
    test_upload = {}

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
test_image_paths = [Path(n) for n in test_upload if Path(n).suffix.lower() in IMAGE_EXTS]
if not test_image_paths:
    # val split can end up empty for a very small dataset; fall back to train images then.
    for split in ("val", "train"):
        candidates = sorted(Path(f"dataset/images/{split}").glob("*"))
        if candidates:
            test_image_paths = candidates[:5]
            print(f"No images uploaded — using {len(test_image_paths)} image(s) from the {split} split instead.")
            break
else:
    print(f"Using {len(test_image_paths)} uploaded image(s).")

In [ ]:
import json

from app.services import detect_service, report_service

DETECT_CONFIDENCE = 0.25  # adjust if you want more/fewer detections in the JSON

images_out = []
for img_path in test_image_paths:
    result = detect_service.run_detection_with_model(
        model, "segment", img_path.read_bytes(), DETECT_CONFIDENCE
    )
    images_out.append(
        {
            "image": str(img_path),
            "image_width": result.image_width,
            "image_height": result.image_height,
            "detections": [report_service.shape_to_report(b) for b in result.boxes],
        }
    )
    print(f"{img_path.name}: {len(result.boxes)} detection(s)")

output = {
    "model": "just-trained best.pt",
    "confidence_threshold": DETECT_CONFIDENCE,
    "images": images_out,
}
Path("colab-detections.json").write_text(json.dumps(output, indent=2))
print("Wrote colab-detections.json")

In [ ]:
from google.colab import files

files.download("colab-detections.json")

In [ ]:
import shutil
from google.colab import files

shutil.copy("runs/train/weights/best.pt", "best.pt")
shutil.make_archive("trained-model", "zip", ".", "best.pt")
files.download("trained-model.zip")